# curl, Requests, XPath, and CSS selectors

Setup

In [ ]:
%%capture output
%%bash
apt-get update
apt-get install -y tree telnet netcat-openbsd


In [ ]:
import gzip
import pandas as pd
import requests
import io
import json
import bs4
from lxml import html as xhtml


## Resources

- [Introduction to web scraping]( https://carpentries-incubator.github.io/lc-webscraping/ )

  - [Selecting content on a web page with XPath]( https://carpentries-incubator.github.io/lc-webscraping/02-xpath/index.html )

  - [Web scraping using Python and Scrapy]( https://carpentries-incubator.github.io/lc-webscraping/04-scrapy/index.html )

- [Playwright]( https://playwright.dev/python/ )

- [HTML Scraping](https://python-docs.readthedocs.io/en/latest/scenarios/scrape.html)

- [Parsing HTML with Beautiful Soup 4]( https://automatetheboringstuff.com/3e/chapter13.html#:~:text=Parsing%20HTML%20with%20Beautiful%20Soup )

  - [Sample files as zip]( https://nostarch.com/download/Automate_the_Boring_Stuff_onlinematerials_v.2.zip )

- [CSS Selector notation]( https://www.w3schools.com/cssref/css_selectors.php )



## Hyper Text Markup Language ( HTML )



"HTML describes the structure of a web page semantically and originally included cues for its appearance."

Features:
- Text with "markup"
- markup == HTML elements/tags
- most tags paired, <html> ... </html>
- nested: tags within tags forming a hierarchy or tree

The term was coined by [Ted Nelson]( https://en.wikipedia.org/wiki/Ted_Nelson ) around 1963 and implemented by [Tim Berners-Lee]( https://en.wikipedia.org/wiki/Tim_Berners-Lee ) in 1989.

A simple [markup example.]( https://en.wikipedia.org/wiki/HTML#Markup )





## Viewing HTML from the browser



### Example.com



One way:
1. View the webpage at http://www.example.com
1. View the source ( Ctrl+U )

Another way:
1. Open Developer tools ( Ctrl+Shift+I )
1. Click on the Elements tab
1. Alt+click or right+click on the "<html>" tag and select "Expand Recursively"

Locating individual elements:
1. Click on the "Select an Element" arrow ( Ctrl+Shift+C )
1. Click on the text "More information..."

Notice that the corresponding section of HTML is highlighted.  You can also click on a section of HTML and the corresponding element in the page will be highlighted.





### ABQ Library



One way:
1. Visit [ABQ databases]( https://abqlibrary.org/az.php?p=1 )
1. View the source ( Ctrl+U )

Another way:
1. Open Developer tools ( Ctrl+Shift+I )
1. Click on the Elements tab
1. Alt+click or right+click on the "<html>" tag and select "Expand Recursively"



## From the command line





In [ ]:
!curl http://www.example.com


In [ ]:
!curl -s -v https://abqlibrary.org/az.php?p=1 2>&1 | cat -n

In [ ]:
!curl -s -v https://abqlibrary.org/az/databases 2>&1 | cat -n

In [ ]:
!curl -s -v -L https://abqlibrary.org/az.php?p=1 2>&1 | cat -n

## Hyper Text Transfer Protocol ( HTTP )



From [Wikipedia: HTTP]( https://en.wikipedia.org/wiki/HTTP )

"The Hypertext Transfer Protocol (HTTP) is an application layer protocol in the Internet protocol suite model for distributed, collaborative, hypermedia information systems."

"In 1991, the first documented official version of HTTP ..., HTTP/0.9, supported only GET method, allowing clients to only retrieve HTML documents from the server, but not supporting any other file formats or information upload."

Features:
- Data flows as a Request/Response pair
- Connections are "stateless", i.e. the server doesn't remember previous requests/responses.
- Requests contain a "verb" and key:value pairs in a Header, and sometimes a payload.
- Responses contain a "status" and key:value pairs in a Header, and often a payload.




Although the protocol has a number of methods/verbs, we will be primarily using GET, HEAD, and POST methods.
- [HTTP methods]( https://en.wikipedia.org/wiki/HTTP#Request_methods )

### Sequence Diagrams of HTTP

- [HTTP sequence diagrams]( https://gist.github.com/rwcitek/397849af2eb90d59e9b6f67d89065a24 )

### Viewing request/response pair from the browser


1. View the webpage at http://www.example.com
1. Open Developer tools ( Ctrl+Shift+I )
1. Click on the Network tab
1. Refresh the page
1. Under the Name field, click on "www.example.com"
1. Scroll down to see the General, Request, and Response sections.
1. Click on the Raw checkbox.
1. Click on the Response tab to view the response payload.







### Viewing request/response pair with curl


Requests have a "> " at the beginning of the line.

Responses have a "< " at the beginning of the line.

In [ ]:
!curl --help


In [ ]:
!curl -s -v http://www.example.com 2>&1 | cat -n

### Request using telnet



In [ ]:
%%bash
telnet www.example.com 80

```text
GET / HTTP/1.1
Host: www.example.com
User-Agent: telnet
Accept: */*
```




### Request using netcat


In [ ]:
%%bash

nc example.com 80 << 'EOF'
GET / HTTP/1.1
Host: example.com
User-Agent: netcat
Connection: close

EOF


### Request using socat


In [ ]:
%%bash

socat stdio,ignoreeof tcp:example.com:80 << 'EOF'
GET / HTTP/1.1
Host: www.example.com
User-Agent: socat
Connection: close

EOF


### Request using raw tcp


In [ ]:
%%bash

exec 3<>/dev/tcp/example.com/80

cat <<EOF >&3
GET / HTTP/1.1
Host: example.com
Connection: close

EOF

cat <&3
exec 3>&-


## Using Python

### Fetching files/pages using the requests module

In [ ]:
page = requests.get('http://www.example.com')


In [ ]:
dict(page.headers)


In [ ]:
page.status_code, page.reason


In [ ]:
[
page.request.url,
page.request.method,
page.request.path_url,
page.request.headers,
page.request.body,
]

In [ ]:
page.url


In [ ]:
page.connection


In [ ]:
page.cookies


In [ ]:
page.text


In [ ]:
page.content


In [ ]:
type(page)

In [ ]:
ls -la

## Trees


From [Wikipedia: Tree (Graph)]( https://en.wikipedia.org/wiki/Tree_(graph_theory) )

"In graph theory, a tree is an undirected graph in which any two vertices are connected by exactly one path, or equivalently a connected acyclic undirected graph."



![Tree]( https://miro.medium.com/v2/resize:fit:714/1*fZn1qW-gXqQImkNuN1_aUg.png "tree image" )



Structure:
- Type of graph, i.e. nodes and edges
- Zero or one parent node
- Zero or more children nodes

Nomenclature:
- Nodes
- Edges
- Directed edge
- Levels, up, down
- Parent, child, sibling
- Ancestors, descendants
- Root ( no parent )
- Branches ( parents and children )
- Leaves ( no children )
- Path: sequence of nodes and edges between two nodes
- Traversing, recursing; depth first, breadth first

Type of trees:
- Binary: at most two children
- Balanced: equal number of levels for all children

Examples:
- Filesystem
- Org charts
- Family pedigrees
- HTML, XML, JSON, YAML

Example: `tree` command for viewing the filesystem

In [ ]:
!tree /etc/apt

In [ ]:
!find /etc/apt

## Beautiful Soup and CSS selectors


In [ ]:
%%capture
%%bash
curl -s -L -o example.html https://autbor.com/example3.html


In [ ]:
pwd

In [ ]:
ls -l

In [ ]:
!cat -n example.html

In [ ]:
with open('example.html') as exampleFile:
  html = exampleFile.read()

exampleSoup = bs4.BeautifulSoup( html, 'html.parser')
print(html)


In [ ]:
html

In [ ]:
type(html)

In [ ]:
html[:10]

In [ ]:
for i, line in enumerate(html.split('\n')):
  print(i+1, line)

In [ ]:
type(exampleSoup)

In [ ]:
elems = exampleSoup.select('#author')
elems

In [ ]:
exampleSoup.select('#login_user')

In [ ]:
type(elems) # elems is a list of Tag objects.

In [ ]:
len(elems)

In [ ]:
elems[0]

In [ ]:
type(elems[0])

In [ ]:
str(elems[0]) # The Tag object as a string.

In [ ]:
elems[0].getText()

In [ ]:
elems[0].attrs

In [ ]:
pElems = exampleSoup.select('p')
pElems

In [ ]:
len(pElems)

In [ ]:
pElems[:2]

In [ ]:
pElems[0]

In [ ]:
type(pElems[0])

In [ ]:
str(pElems[0])

In [ ]:
pElems[0].getText()

In [ ]:
str(pElems[1])

In [ ]:
pElems[1].getText()

In [ ]:
str(pElems[2])

In [ ]:
pElems[2].getText()

In [ ]:
aElems = exampleSoup.select('a')
aElems


In [ ]:
len(aElems)

In [ ]:
aElems[0]


In [ ]:
aElems[0].attrs


In [ ]:
aElems[0].attrs["href"]


In [ ]:
body_text = exampleSoup.select("body")[0].get_text()
print( body_text )


In [ ]:
body_text.split('\n')


### Using the data set from stamp prices


In [ ]:
url = 'https://vincentarelbundock.github.io/Rdatasets/datasets.html'
url


In [ ]:
datasets_tables = pd.read_html( url )
len(datasets_tables)


In [ ]:
datasets_df = datasets_tables[1]
datasets_df.head()


In [ ]:
datasets_df.shape


In [ ]:
datasets_df.head()["CSV"]

In [ ]:
data_sets = requests.get( url )
data_sets


In [ ]:
html = data_sets.text
html[:1000]


In [ ]:
dom = bs4.BeautifulSoup(html, "html.parser")
str(dom).split("\n")[:10]

In [ ]:
str(dom).split("\n")[:51]


In [ ]:
a_tags = dom.select("a")
a_tags[:6], a_tags[-6:]



In [ ]:
len(a_tags)


In [ ]:
a_tags[100].attrs.get("href","")

In [ ]:
hrefs = [ tag.attrs.get("href","") for tag in a_tags ]
hrefs[:10]


In [ ]:
csvs = []
for x in hrefs:
  if x.endswith("csv"):
    csvs.append(x)

len(csvs)


In [ ]:
csvs = [ x for x in hrefs if x.endswith("csv") ]
len(csvs)


In [ ]:
csvs[:10]

In [ ]:
for csv in csvs[:10]:
  print(f"== {csv}")
  df = pd.read_csv( csv )
  print(df.index.size)


In [ ]:
datasets_df["cvs_url"] = csvs
datasets_df

In [ ]:
pd.read_csv(csvs[1]).index.size


In [ ]:
# datasets_df["row_count"] = [ pd.read_csv(x).index.size for x in csvs ]
# datasets_df


If we get an error, e.g. 503, how should/could we handle it.


In [ ]:
docs = [
  x
    for x in hrefs
      if x.endswith("html")
]

len(docs)


In [ ]:
datasets_df["docs_url"] = docs
datasets_df

### Using DBpedia

In [ ]:
dbpedia = requests.get("https://dbpedia.org/page/Digby_Morrell")
dbpedia

In [ ]:
html = dbpedia.text
html

In [ ]:
dom = bs4.BeautifulSoup(html, "html.parser")
dom

In [ ]:
type(html)

In [ ]:
type(dom)

In [ ]:
a_tags = dom.select("a")
a_tags

In [ ]:
len(a_tags)

In [ ]:
[ tag.attrs for tag in a_tags ]

In [ ]:
for a_tag in a_tags:
  href = a_tag.attrs["href"]
  if href.startswith("http"):
    rev = a_tag.attrs.get("rev","")
    if "foaf:primaryTopic" in rev:
      print(href)


#### Using only CSS selectors.

In [ ]:
foaf_pt = dom.select('a[href^="http"][rev="foaf:primaryTopic"]')
foaf_pt[0].attrs["href"]


In [ ]:
foaf_pt[0].attrs["href"].split("/")[-1]


### link extraction from web page



- [gist for extracting links from a web page]( https://gist.github.com/rwcitek/6a24241fcc61c0fc4055dda77d347c4a )




Example: extracting all the URLs for data from IMDB.

- [IMDB data]( https://datasets.imdbws.com/ )

In [ ]:
url = "https://datasets.imdbws.com/"
soup = bs4.BeautifulSoup(requests.get(url).text, 'html.parser')

imdb_files = [
  tag.get('href')
    for tag in soup.select('a[href$=".tsv.gz"]')
]

imdb_files

## XPath

Using the `lxml` library to pull the same information as the CSS selector.

In [ ]:
page = requests.get('https://autbor.com/example3.html')
tree = xhtml.fromstring(page.content)
tree

/html/body/p[2]/a

In [ ]:
page.content

In [ ]:
full_xpath = '/html/body/p[2]/a'
full_xpath


In [ ]:
title = tree.xpath(full_xpath)
type(title)


In [ ]:
len(title)

In [ ]:
type(title[0])


In [ ]:
type(title[0].text)


In [ ]:
print(title[0].text)


In [ ]:
title = tree.xpath('/html/head/title/text()')
print(title[0])


In [ ]:
link = tree.xpath('/html/body/p[2]/a/@href')
print(link[0])


In [ ]:
with open('example.html') as exampleFile:
  content = exampleFile.read()
print(content)


In [ ]:
# Get an element list
tree = xhtml.fromstring( content )
elems_xp = tree.xpath('//*[@*="author"]')
elems_xp


In [ ]:
# Get one element
elems_xp[0]


In [ ]:
# Text bounded by the tag
elems_xp[0].text

In [ ]:
# Element attributes
elems_xp[0].attrib


In [ ]:
pElems_xp = tree.xpath('//p')
pElems_xp


In [ ]:
pElems_xp[0]


In [ ]:
pElems_xp[0].text


In [ ]:
pElems_xp[1]


In [ ]:
pElems_xp[1].text


In [ ]:
pElems_xp[2]


In [ ]:
pElems_xp[2].text


## CSS vs XPATH


In [ ]:
example = '''
<body>
  <p class="phrase">Hello, world!</p>
  <p>By <span id="name">Foo Bar</span></p>
</body></html>
'''
example


In [ ]:
# Using CSS selectors
tree_css = bs4.BeautifulSoup( example, 'html.parser')
elems_css = tree_css.select('#name')
elems_css

In [ ]:
tree_css

In [ ]:
elems_css[0]

In [ ]:
elems_css[0].text

In [ ]:
str(elems_css[0])

In [ ]:
# Using XPath
tree = xhtml.fromstring( example )
elems_xp = tree.xpath('//*[@*="name"]')
elems_xp


In [ ]:
elems_xp[0]


In [ ]:
# Text bounded by the tag
elems_xp[0].text

In [ ]:
# Can't seem to get the text of the entire tag
str(elems_xp[0])


## HEAD request

Having fun with HTTP headers:
- [Fun and unusual HTTP response headers]( https://www.pingdom.com/blog/fun-and-unusual-http-response-headers/ )

Response codes:
- MDN [HTTP response status codes]( https://developer.mozilla.org/en-US/docs/Web/HTTP/Status )
- Wikipedia [List of HTTP status codes]( https://en.wikipedia.org/wiki/List_of_HTTP_status_codes )


In [ ]:
!curl -L -s -I -XHEAD 'https://jgmsinc.com/'


In [ ]:
!curl -s -I -XHEAD 'https://careers-encantadotech.icims.com/jobs/search?ss=1&searchRelation=keyword_all&searchCategory=8730'


In [ ]:
!curl -s -L -I -XHEAD 'https://cnmingenuity.org/'


In [ ]:
!curl -I -s https://ddc-datascience.s3.amazonaws.com/Projects/Project.1-Transactions/Data/Transaction.train.big.csv


## robots.txt


"robots.txt is the filename used for implementing the Robots Exclusion Protocol, a standard used by websites to indicate to visiting web crawlers and other web robots which portions of the website they are allowed to visit."

- Wikipedia [robots.txt]( https://en.wikipedia.org/wiki/Robots.txt )
- Google docs [Introduction to robots.txt]( https://developers.google.com/search/docs/crawling-indexing/robots/intro )
- [RFC 9309]( https://www.rfc-editor.org/rfc/rfc9309 )




In [ ]:
!curl -s 'https://cnmingenuity.org/robots.txt'


In [ ]:
!curl -s 'https://www.cabq.gov/robots.txt'


## XML



### Sitemaps




- Wikipedia [sitemaps]( https://en.wikipedia.org/wiki/Sitemaps )



In [ ]:
!zcat --help

#### Read the sitemap index XML

In [ ]:
sitemap_url = 'https://www.cabq.gov/sitemap.xml.gz'
sitemap_url

In [ ]:
!curl -s {sitemap_url} | zcat


Read using Pandas


In [ ]:
response = requests.get(sitemap_url, stream=True)
response

In [ ]:
with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as f:
    sitemap_index_df = pd.read_xml(f)


In [ ]:
sitemap_index_df

#### Read the sitemap XML files

In [ ]:
!curl -s 'https://www.cabq.gov/sitemap1.xml.gz' | zcat | head -20

Using Pandas


In [ ]:
dfs = []
for sitemap_url in sitemap_index_df["loc"]:
  response = requests.get(sitemap_url, stream=True)
  with gzip.GzipFile(fileobj=io.BytesIO(response.content)) as f:
    dfs.append(pd.read_xml(f))
sitemap_df = pd.concat(dfs, ignore_index=True)
sitemap_df.shape

In [ ]:
sitemap_df

#### HEAD of random three items in sitemap file

In [ ]:
for url in sitemap_df.sample(n=3, random_state=42)["loc"]:
  response = requests.head(url)
  print(f"== {url}")
  print(f"{response.status_code} {response.reason}")
  for k,v in response.headers.items():
    print(f"{k}: {v}")
  print()


#### A missing entry


In [ ]:
url = 'https://www.cabq.gov/transit/news/abq-ride-providing-shuttle-service-around-detours-at-the-twinkle-light-parade'
url


In [ ]:
filter = ( sitemap_df["loc"] == url )
sitemap_df[ filter ]


In [ ]:
response = requests.head(url)
print(f"== {url}")
print(f"{response.status_code} {response.reason}")
for k,v in response.headers.items():
  print(f"{k}: {v}")
print()


#### Plot distribution of file types

In [ ]:
(
sitemap_df["loc"]
    .str.split(".")
    .str[-1]
    .str.lower()
    .value_counts()
    .head(20)
    .sort_values(ascending = True)
    .plot( kind = "barh" )
) ;